# 🔍 Text Embeddings & FAISS FAISS Prototype

This notebook takes the flattened Shopify product data from Notebook 01, generates semantic vector embeddings using a local HuggingFace model (`sentence-transformers`), and indexes them into FAISS for fast similarity search.

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import time

### 1. Load Data & Initialize Embedding Model

We use `all-MiniLM-L6-v2` as it is extremely fast and light enough to run on a CPU locally while providing decent semantic matching.

In [ ]:
# Load data from previous step
df = pd.read_csv("shopify_products_prep.csv")
print(f"Loaded {len(df)} products.")

# Initialize Sentence Transformer
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

### 2. Generate Embeddings

We embed that rich text string we created (`embedding_text`).

In [ ]:
texts_to_embed = df["embedding_text"].tolist()

print(f"Generating embeddings for {len(texts_to_embed)} products...")
start_time = time.time()

embeddings = model.encode(texts_to_embed, show_progress_bar=True)
embeddings = np.array(embeddings).astype("float32") # FAISS requires float32

print(f"Done in {time.time() - start_time:.2f} seconds.")
print(f"Embedding Shape: {embeddings.shape}")

### 3. Build the FAISS Index

We use `IndexFlatL2` for exact nearest neighbor search using Euclidean distance.

In [ ]:
dimension = embeddings.shape[1]

# Create Index
index = faiss.IndexFlatL2(dimension)

# Add vectors to the index
index.add(embeddings)
print(f"FAISS Index contains {index.ntotal} vectors.")

### 4. Semantic Search Function

Now we can search our products using natural language!

In [ ]:
def search_products(query, k=5):
    # Embed the user query
    query_vector = model.encode([query]).astype("float32")
    
    # Search FAISS
    distances, indices = index.search(query_vector, k)
    
    # Map back to our DataFrame
    results = []
    for i, idx in enumerate(indices[0]):
        if idx != -1:  # -1 means no result found
            row = df.iloc[idx]
            results.append({
                "score": distances[0][i],
                "title": row["title"],
                "vendor": row["metadata"] if pd.notna(row["metadata"]) else "",
                "text": row["embedding_text"]
            })
            
    return pd.DataFrame(results)

In [ ]:
# Test out semantic searches that native Shopify can't do well:
queries = [
    "summer outdoor activewear",
    "something cozy for winter",
    "formal event attire"
]

for q in queries:
    print(f"\n{'='*40}\nQuery: '{q}'\n{'='*40}")
    res = search_products(q, k=3)
    display(res[["score", "title"]])